In [2]:
import torch
import math
from torch import nn

In [4]:
x = torch.rand(128,32,512)
d_model = 512
n_head = 8

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,n_head):
        super().__init__()
        self.n_head = n_head                                        # 多头数量（比如8头）
        self.d_model = d_model                                      # 模型向量维度（比如512
        # self.d_k = d_model // n_head                              # 每个头的维度
        self.w_q = nn.Linear(d_model, d_model)                      # qkv线性层，把输入向量做线性变换，得到Q（查询），K(键)，V（值）                    
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_combine = nn.Linear(d_model, d_model)                # 输出线性层，把多头拼回一个向量
        self.softmax = nn.Softmax(dim=-1)                           # 注意力权重归一化

    def forward(self,q,k,v,mask=None):
        batch, time, dimension = q.shape                            # 批次，句子长度，向量维度
        n_d = self.d_model // self.n_head                           # 
        q, k, v = self.w_q(q), self.w_k(k), self.w_v(v)             # qkv线性变换
        q = q.view(batch,time,self.n_head,n_d).permute(0,2,1,3)     # 拆分多头，把[batch, ]
        k = k.view(batch,time,self.n_head,n_d).permute(0,2,1,3)
        v = v.view(batch,time,self.n_head,n_d).permute(0,2,1,3)
        score = q @ k.transpose(2,3) / math.sqrt(n_d)         # 计算注意力分数
        if mask is not None:
            score = score.masked_fill(mask==0,-1e9)
        score = self.softmax(score) @ v
        score = score.permute(0,2,1,3).contiguous().view(batch,time,dimension)  # 拼接多头
        out = self.w_combine(score)
        return out
    

attention = MultiHeadAttention(d_model, n_head)





In [6]:
out = attention(x,x,x)
print(out)

RuntimeError: shape '[128, 32, 8, 64]' is invalid for input of size 32768